In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [6]:
# 1. Load dataset
df = pd.read_csv("../synthatic/USA_Housing_Synthetic_10k.csv")
df = df.dropna().copy() 


In [7]:
print("Dataset shape:", df.shape)
print("\nFirst five rows:")
print(df.head())

Dataset shape: (10000, 7)

First five rows:
   Avg. Area Income  Avg. Area House Age  Avg. Area Number of Rooms  \
0          73484.91                 5.82                       4.54   
1          49280.29                 7.12                       7.09   
2          81508.12                 2.85                       6.14   
3          84930.16                 4.99                       5.60   
4          32881.37                 6.99                       5.95   

   Avg. Area Number of Bedrooms  Area Population      Price  \
0                          1.29         27902.09  556616.99   
1                          2.39         24676.45  462160.71   
2                          1.63         38993.55  525202.82   
3                          1.68         37380.09  508545.37   
4                          1.94         15978.97  636592.87   

                                         Address  
0       Synthetic Property 00001, Boise, ID, USA  
1    Synthetic Property 00002, Portland, OR, USA

# Price is the target variable. Address is excluded from the predictors.

In [8]:
# Price is the target variable. Address is excluded from the predictors.
target = "Price"
all_features = [
    "Avg. Area Income",
    "Avg. Area House Age",
    "Avg. Area Number of Rooms",
    "Avg. Area Number of Bedrooms",
    "Area Population"
]
three_features = [
    "Avg. Area Income",
    "Avg. Area House Age",
    "Avg. Area Number of Rooms"
]

In [9]:
X_all = df[all_features]
X_three = df[three_features]
y = df[target]

In [10]:
print("\nQuestion 1 features:", all_features)
print("Question 2 features:", three_features)



Question 1 features: ['Avg. Area Income', 'Avg. Area House Age', 'Avg. Area Number of Rooms', 'Avg. Area Number of Bedrooms', 'Area Population']
Question 2 features: ['Avg. Area Income', 'Avg. Area House Age', 'Avg. Area Number of Rooms']


# Split, normalize, train and evaluate

In [12]:
def evaluate_model(X, y, random_state=42, show_details=True):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=random_state
    )

    # StandardScaler is fitted on training data only.
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("regression", LinearRegression())
    ])
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, predictions)

    if show_details:
        scaler = model.named_steps["scaler"]
        reg = model.named_steps["regression"]

        # Convert coefficients back to original feature units.
        coefficients = reg.coef_ / scaler.scale_
        intercept = reg.intercept_ - np.sum(
            reg.coef_ * scaler.mean_ / scaler.scale_
        )

        print("\nRandom state:", random_state)
        print(f"Beta 0 (intercept): {intercept:.4f}")
        print("\nRegression coefficients:")
        for feature, beta in zip(X.columns, coefficients):
            print(f"Beta for {feature}: {beta:.4f}")

        print("\nEvaluation metrics:")
        print(f"MAE  : {mae:,.4f}")
        print(f"MSE  : {mse:,.4f}")
        print(f"RMSE : {rmse:,.4f}")
        print(f"R2   : {r2:.6f}")

    return {"MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2}



# Question 1: MLR using all numerical features except Address

In [13]:
print("\n" + "=" * 65)
print("QUESTION 1: ALL FEATURES EXCEPT ADDRESS")
print("=" * 65)
q1_metrics = evaluate_model(X_all, y, random_state=42)



QUESTION 1: ALL FEATURES EXCEPT ADDRESS

Random state: 42
Beta 0 (intercept): 77704.8949

Regression coefficients:
Beta for Avg. Area Income: 2.1189
Beta for Avg. Area House Age: 18357.7852
Beta for Avg. Area Number of Rooms: 28988.9442
Beta for Avg. Area Number of Bedrooms: 6698.7524
Beta for Area Population: 1.2004

Evaluation metrics:
MAE  : 43,001.4660
MSE  : 2,905,300,446.6365
RMSE : 53,900.8390
R2   : 0.550414


# Question 2: MLR using any three features except Address

In [14]:
print("\n" + "=" * 65)
print("QUESTION 2: THREE SELECTED FEATURES")
print("=" * 65)
q2_metrics = evaluate_model(X_three, y, random_state=42)



QUESTION 2: THREE SELECTED FEATURES

Random state: 42
Beta 0 (intercept): 120556.3498

Regression coefficients:
Beta for Avg. Area Income: 2.1248
Beta for Avg. Area House Age: 18310.0605
Beta for Avg. Area Number of Rooms: 31257.3756

Evaluation metrics:
MAE  : 44,002.6134
MSE  : 3,037,251,353.6682
RMSE : 55,111.2634
R2   : 0.529995


# Question 3: Test 200 random_state values and choose highest R2

In [15]:
def find_best_random_state(X, y, model_name, output_file):
    results = []

    for state in range(200):
        metrics = evaluate_model(
            X, y, random_state=state, show_details=False
        )
        results.append({"random_state": state, **metrics})

    results_df = pd.DataFrame(results)
    best = results_df.loc[results_df["R2"].idxmax()]

    print("\n" + "=" * 65)
    print(model_name)
    print("=" * 65)
    print("Best random_state:", int(best["random_state"]))
    print(f"Best R2: {best['R2']:.6f}")
    print(f"MAE: {best['MAE']:,.4f}")
    print(f"MSE: {best['MSE']:,.4f}")
    print(f"RMSE: {best['RMSE']:,.4f}")

    results_df.to_csv(output_file, index=False)
    print("All 200 results saved to:", output_file)

    return int(best["random_state"]), results_df


In [16]:
best_state_all, results_all = find_best_random_state(
    X_all, y,
    "QUESTION 3: ALL FIVE FEATURES",
    "q1_all_features_200_splits.csv"
)

best_state_three, results_three = find_best_random_state(
    X_three, y,
    "QUESTION 3: THREE FEATURES",
    "q2_three_features_200_splits.csv"
)



QUESTION 3: ALL FIVE FEATURES
Best random_state: 25
Best R2: 0.578397
MAE: 42,694.0915
MSE: 2,864,173,576.3348
RMSE: 53,517.9743
All 200 results saved to: q1_all_features_200_splits.csv

QUESTION 3: THREE FEATURES
Best random_state: 25
Best R2: 0.558009
MAE: 43,942.7786
MSE: 3,002,677,320.7897
RMSE: 54,796.6908
All 200 results saved to: q2_three_features_200_splits.csv


# Question 4: Normalization
# StandardScaler normalization is applied in the Pipeline above.

In [17]:
print("\n" + "=" * 65)
print("QUESTION 4: FEATURE NORMALIZATION")
print("=" * 65)
print("StandardScaler formula: z = (x - mean) / standard deviation")
print("Scaler is fitted on training data only to prevent data leakage.")




QUESTION 4: FEATURE NORMALIZATION
StandardScaler formula: z = (x - mean) / standard deviation
Scaler is fitted on training data only to prevent data leakage.


# Final comparison

In [18]:
comparison = pd.DataFrame([
    {
        "Model": "All five features except Address",
        "Best random_state": best_state_all,
        "Best R2": results_all["R2"].max()
    },
    {
        "Model": "Three selected features",
        "Best random_state": best_state_three,
        "Best R2": results_three["R2"].max()
    }
])

print("\nFINAL MODEL COMPARISON")
print(comparison.to_string(index=False))

if results_all["R2"].max() > results_three["R2"].max():
    print("\nConclusion: The all-feature model has the higher R2 score.")
elif results_all["R2"].max() < results_three["R2"].max():
    print("\nConclusion: The three-feature model has the higher R2 score.")
else:
    print("\nConclusion: Both models have the same best R2 score.")

print("\nLab 5 completed successfully.")



FINAL MODEL COMPARISON
                           Model  Best random_state  Best R2
All five features except Address                 25 0.578397
         Three selected features                 25 0.558009

Conclusion: The all-feature model has the higher R2 score.

Lab 5 completed successfully.
